In [ ]:
import numpy as np
import tensorlayerx as tlx
from tensorlayerx.dataflow import Dataset, DataLoader
from tensorlayerx.vision import transforms, load_images
## enable debug logging
tlx.logging.set_verbosity(tlx.logging.DEBUG)

class FLAGS(object):
    def __init__(self):
        self.n_epoch = 25 # "Epoch to train [25]"
        self.z_dim = 100 # "Num of noise value]"
        self.lr = 0.0002 # "Learning rate of for adam [0.0002]")
        self.beta1 = 0.5 # "Momentum term of adam [0.5]")
        self.batch_size = 64 # "The number of batch images [64]")
        self.output_size = 64 # "The size of the output images to produce [64]")
        self.sample_size = 64 # "The number of sample images [64]")
        self.c_dim = 3 # "Number of image channels. [3]")
        self.save_every_epoch = 1 # "The interval of saveing checkpoints.")
        # self.dataset = "celebA" # "The name of dataset [celebA, mnist, lsun]")
        self.checkpoint_dir = "checkpoint" # "Directory name to save the checkpoints [checkpoint]")
        self.sample_dir = "samples" # "Directory name to save the image samples [samples]")
        assert np.sqrt(self.sample_size) % 1 == 0., 'Flag `sample_size` needs to be a perfect square'
flags = FLAGS()

tlx.files.exists_or_mkdir(flags.checkpoint_dir) # save model
tlx.files.exists_or_mkdir(flags.sample_dir) # save generated image

transforms_celebA = transforms.Compose(
    [
        transforms.CentralCrop(size = [128, 128]),
        transforms.Resize(size=64),
        transforms.RandomFlipHorizontal(),
        transforms.Normalize(mean=(127.5), std=(127.5), data_format='HWC'),
    ]
)

class CELEBA(Dataset):

    def __init__(self):
        images_path = r'data/celebA/img_align_celeba'
        self.images = load_images(images_path, n_threads=0)

    def __getitem__(self, idx):
        image = self.images[idx]
        image = transforms_celebA(image)
        return image

    def __len__(self):
        return len(self.images)


def get_celebA(batch_size):
    # dataset API and augmentation
    images_path = tlx.files.load_celebA_dataset()
    celebA = CELEBA()
    trainloader = DataLoader(celebA, batch_size=batch_size, shuffle=True, drop_last=True)
    return trainloader, images_path


In [ ]:
import tensorlayerx as tlx
from tensorlayerx.nn import Linear, ConvTranspose2d, Reshape, BatchNorm2d, Conv2d, Flatten, Module

class Generator(Module):
    gf_dim = 64
    image_size = 64
    s16 = image_size // 16
    w_init = tlx.nn.initializers.random_normal(stddev=0.02)
    gamma_init = tlx.nn.initializers.random_normal(1., 0.02)

    def __init__(self):
        super(Generator, self).__init__()
        self.linear1 = Linear(out_features=self.gf_dim * 8 * self.s16 * self.s16,  W_init=self.w_init, b_init=None)
        self.reshape = Reshape(shape=(-1, self.s16, self.s16, self.gf_dim * 8))
        self.bn1 = BatchNorm2d(0.9, act=tlx.nn.ReLU, gamma_init=self.gamma_init)
        self.deconv2d1 = ConvTranspose2d(self.gf_dim * 4, (5, 5), (2, 2), W_init=self.w_init, b_init=None)
        self.bn2 = BatchNorm2d(0.9, act=tlx.nn.ReLU, gamma_init=self.gamma_init)
        self.deconv2d2 = ConvTranspose2d(self.gf_dim * 2, (5, 5), (2, 2), W_init=self.w_init, b_init=None)
        self.bn3 = BatchNorm2d(0.9, act=tlx.nn.ReLU, gamma_init=self.gamma_init)
        self.deconv2d3 = ConvTranspose2d(self.gf_dim, (5, 5), (2, 2), W_init=self.w_init, b_init=None)
        self.bn4 = BatchNorm2d(0.9, act=tlx.nn.ReLU, gamma_init=self.gamma_init)
        self.deconv2d4 = ConvTranspose2d(3, (5, 5), (2, 2), act=tlx.ops.tanh, W_init=self.w_init)

    def forward(self, x):
        x = self.linear1(x)
        x = self.reshape(x)
        x = self.bn1(x)
        x = self.deconv2d1(x)
        x = self.bn2(x)
        x = self.deconv2d2(x)
        x = self.bn3(x)
        x = self.deconv2d3(x)
        x = self.bn4(x)
        x = self.deconv2d4(x)

        return x

class Discriminator(Module):

    df_dim = 64
    w_init = tlx.nn.initializers.random_normal(stddev=0.02)
    gamma_init = tlx.nn.initializers.random_normal(1., 0.02)

    def __init__(self):
        super(Discriminator, self).__init__()
        self.conv1 = Conv2d(self.df_dim, (5, 5), (2, 2), act=tlx.nn.LeakyReLU(0.2), W_init=self.w_init)
        self.conv2 = Conv2d(self.df_dim * 2, (5, 5), (2, 2), W_init=self.w_init, b_init=None)
        self.bn1 = BatchNorm2d(0.9, act=tlx.nn.LeakyReLU(0.2), gamma_init=self.gamma_init)
        self.conv3 = Conv2d(self.df_dim * 4, (5, 5), (2, 2), W_init=self.w_init, b_init=None)
        self.bn2 = BatchNorm2d(0.9, act=tlx.nn.LeakyReLU(0.2), gamma_init=self.gamma_init)
        self.conv4 = Conv2d(self.df_dim * 8, (5, 5), (2, 2), W_init=self.w_init, b_init=None)
        self.bn3 = BatchNorm2d(0.9, act=tlx.nn.LeakyReLU(0.2), gamma_init=self.gamma_init)
        self.flatten = Flatten()
        self.linear = Linear(1, W_init=self.w_init)


    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.bn1(x)
        x = self.conv3(x)
        x = self.bn2(x)
        x = self.conv4(x)
        x = self.bn3(x)
        x = self.flatten(x)
        x = self.linear(x)
        return x

In [ ]:
import os
# os.environ['TL_BACKEND'] = 'tensorflow' # Just modify this line, easily switch to any framework!
# os.environ['TL_BACKEND'] = 'mindspore'
# os.environ['TL_BACKEND'] = 'paddle'
os.environ['TL_BACKEND'] = 'torch'
import time
import numpy as np
import tensorlayerx as tlx
from tensorlayerx.nn import Module
from tensorlayerx.utils.visualize import save_images
from tensorlayerx.model import TrainOneStep
from data import get_celebA, flags
from model import  Generator, Discriminator
# tlx.set_device('GPU') # use this ops set default device.
num_tiles = int(np.sqrt(flags.sample_size))

class WithLoss_D(Module):
    def __init__(self, D, G):
        super(WithLoss_D, self).__init__()
        self.D = D
        self.G = G

    def forward(self, images, fake):
        d_logits = self.D(self.G(fake))
        d2_logits = self.D(images)
        d_loss_real = tlx.losses.sigmoid_cross_entropy(d2_logits, tlx.ones_like(d2_logits))
        # discriminator: images from generator (fake) are labelled as 0
        d_loss_fake = tlx.losses.sigmoid_cross_entropy(d_logits, tlx.zeros_like(d_logits))
        d_loss = d_loss_real + d_loss_fake
        return d_loss

class WithLoss_G(Module):
    def __init__(self, D, G):
        super(WithLoss_G, self).__init__()
        self.D = D
        self.G = G

    def forward(self, images, fake):
        d_logits = self.D(self.G(fake))
        g_loss = tlx.losses.sigmoid_cross_entropy(d_logits, tlx.ones_like(d_logits))
        return g_loss

def train():
    images_loader, images_path = get_celebA(flags.batch_size)
    G = Generator()
    D = Discriminator()
    G.init_build(tlx.nn.Input(shape=(flags.batch_size, 100)))
    D.init_build(tlx.nn.Input(shape=(flags.batch_size, 64, 64, 3)))

    G.set_train()
    D.set_train()

    d_optimizer = tlx.optimizers.Adam(flags.lr, beta_1=flags.beta1)
    g_optimizer = tlx.optimizers.Adam(flags.lr, beta_1=flags.beta1)

    g_weights = G.trainable_weights
    d_weights = D.trainable_weights


    net_with_loss_D = WithLoss_D(D, G)
    net_with_loss_G = WithLoss_G(D, G)
    trainforG = TrainOneStep(net_with_loss_G, optimizer=g_optimizer, train_weights=g_weights)
    trainforD = TrainOneStep(net_with_loss_D, optimizer=d_optimizer, train_weights=d_weights)

    n_step_epoch = int(len(images_path) // flags.batch_size)

    # Z = tf.distributions.Normal(0., 1.)
    for epoch in range(flags.n_epoch):
        for step, batch_images in enumerate(images_loader):
            step_time = time.time()
            z = np.random.normal(loc=0.0, scale=1.0, size=[flags.batch_size, flags.z_dim]).astype(np.float32)
            z = tlx.ops.convert_to_tensor(z)
            d_loss = trainforD(batch_images, z)
            g_loss = trainforG(batch_images, z)

            print("Epoch: [{}/{}] [{}/{}] took: {:.3f}, d_loss: {:.5f}, g_loss: {:.5f}".format(epoch, \
                  flags.n_epoch, step, n_step_epoch, time.time()-step_time, float(d_loss), float(g_loss)))

        if np.mod(epoch, flags.save_every_epoch) == 0:
            G.save_weights('{}/G.npz'.format(flags.checkpoint_dir), format='npz')
            D.save_weights('{}/D.npz'.format(flags.checkpoint_dir), format='npz')
            G.set_eval()
            result = G(z)
            G.set_train()
            save_images(tlx.convert_to_numpy(result), [num_tiles, num_tiles], '{}/train_{:02d}.png'.format(flags.sample_dir, epoch))

if __name__ == '__main__':
    train()